In [5]:
import numpy as np
from scipy.optimize import minimize

# Coordinates extracted from Task6.png for Q2
CAR_POSE_INIT = (0.1, 3.1, np.arctan2(1.1, 0.1))  # (x, y, theta [rad])
WP_PREV = (0.9, 3.0)                              # Previous waypoint (x1, y1)
WP_NEXT = (1.0, 4.1)                              # Next waypoint (x2, y2)

v0 = 5.5
v_des = 10.0
dt = 0.02
N_steps = 3
u_init = [1.3, 0.23]

# Weights
W_h = .1
W_ct = 0.5
W_v = 0.01

x1, y1 = WP_PREV
x2, y2 = WP_NEXT
dx, dy = x2 - x1, y2 - y1
line_len = np.hypot(dx, dy)
theta_path = np.arctan2(dy, dx)

def wrap_angle(angle):
    return (angle + np.pi) % (2.0 * np.pi) - np.pi

def cost_function(u):
    a, omega = u
    x, y, theta = CAR_POSE_INIT
    v = v0
    total_cost = 0.0
    
    for _ in range(N_steps):
        s = v * dt + 0.5 * a * (dt ** 2)
        theta_next = theta + omega * dt
        theta_avg = 0.5 * (theta + theta_next)
        
        x += s * np.cos(theta_avg)
        y += s * np.sin(theta_avg)
        theta = theta_next
        v += a * dt
        
        e_h = wrap_angle(theta - theta_path)
        e_ct = ((dx * (y1 - y)) - ((x1 - x) * dy)) / line_len
        e_v = v - v_des
        
        total_cost += (W_h * (e_h ** 2)) + (W_ct * (e_ct ** 2)) + (W_v * (e_v ** 2))
        
    return total_cost

# Optimisation
J_initial = cost_function(u_init)
bounds = [(-10.0, 4.5), (-1.5, 1.5)]

res_bounded = minimize(cost_function, u_init, method='SLSQP', bounds=bounds)
res_unbounded = minimize(cost_function, u_init, method='BFGS')

print(f"J Initial: {J_initial:.4f}")
print(f"[Bounded]   a* = {res_bounded.x[0]:.4f}, omega* = {res_bounded.x[1]:.4f}, J* = {res_bounded.fun:.4f}")
print(f"[Unbounded] a* = {res_unbounded.x[0]:.4f}, omega* = {res_unbounded.x[1]:.4f}, J* = {res_unbounded.fun:.4f}")

J Initial: 10.3616
[Bounded]   a* = 4.5000, omega* = -1.5000, J* = 10.1219
[Unbounded] a* = 158.1644, omega* = -13.7206, J* = 8.3562
